# Phyllome CEA — Crop Growth & Financial Viability Pipeline
**Author:** Diego Fernández G. — Master by Research, University of Sydney  
**Data:** Phyllome vertical farm, Jun 2024 – Dec 2025  
**Goal:** Calibrate crop growth models and bridge them to a financial viability
framework for Controlled-Environment Agriculture (CEA) evaluation.

---
## Contents
1. Setup & data loading
2. Physical coordinate system & sensor positions
3. Microclimate assignment (anisotropic IDW interpolation)
4. Empirical yield models (OLS + Random Forest with LOSO-CV)
5. Biomass growth curves (Gompertz)
6. Mechanistic growth model (Vanthoor, simplified)
7. Summary of calibrated parameters
8. Financial bridge module — R/m²/year viability analysis
9. Operating window optimisation — yield per kWh & sensitivity


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.optimize import curve_fit, minimize
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# ── Paths (adjust to your local clone)
BASE    = "data/"       # cleaned CSVs
UPLOADS = "data/raw/"   # original source files

# ── Global plot style
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# ── PPFD colour palette (warm — consistent with project aesthetics)
PPFD_COLORS = {180.0: '#4C9BE8', 310.0: '#F4A020'}

# ── Financial-bridge accent colours
COL_VIABLE   = '#2E7D32'   # dark green — viable
COL_MARGINAL = '#E9C46A'   # sand gold — marginal
COL_LOSS     = '#E63946'   # red — loss

print("Libraries loaded ✓")


In [ ]:
# Cleaned datasets (produced by data_cleaning.py)
air   = pd.read_csv(f"{BASE}air_clean.csv",    parse_dates=['timestamp'])
water = pd.read_csv(f"{BASE}water_clean.csv",  parse_dates=['timestamp'])
prod  = pd.read_csv(f"{BASE}production_clean.csv",
                    parse_dates=['growth_start','harvest_date'])
mc    = pd.read_csv(f"{BASE}tray_microclimate.csv",
                    parse_dates=['growth_start','harvest_date'])

print(f"Air sensor data   : {len(air):,} rows | {air['deviceId'].nunique()} sensors")
print(f"Water sensor data : {len(water):,} rows")
print(f"Production data   : {len(prod):,} trays | {prod['Season'].nunique()} seasons")
print(f"Tray microclimate : {len(mc):,} valid trays with env. conditions assigned")
print()
print("Yield summary (g/tray):")
print(mc['yield_g'].describe().round(1).to_string())


## 2. Physical Coordinate System & Sensor Positions

### Grow room geometry
- **4 columns**, **12 shelves**, **18 rows** per column  
- Columns 1 & 2: same rack, back-to-back (20 mm separation)  
- Columns 3 & 4: same rack, back-to-back (20 mm separation)  
- Aisle between rack pairs: 1 140 mm  
- Tray size: 1 155 × 1 155 mm  
- Row pitch: 1 175 mm | Shelf pitch: 500 mm | Shelf 1 at 700 mm from floor  

### Sensors
9 sensors (deviceId) in columns 1 and 3 only.  
Column 4 systematically ~7 pp drier than column 3 → corrected post-interpolation.  
Sensor c44c (col 1, shelf 1): CO₂ data removed entirely (malfunctioning throughout).


In [ ]:
# Physical coordinate mapping
COL_X   = {1: 0, 2: 20, 3: 1160, 4: 1180}   # mm (x-axis)
ROW_Y   = lambda r: (r - 1) * 1175            # mm (y-axis)
SHELF_Z = lambda s: 700 + (s - 1) * 500       # mm (z-axis)

SENSORS = [
    ('c44cba6d9fe8', 1,  1,  3, False),
    ('484cba6d9fe8', 1,  8,  3, True),
    ('104dba6d9fe8', 1, 13,  3, True),
    ('884cba6d9fe8', 3,  2, 18, True),
    ('744cba6d9fe8', 3,  3,  3, True),
    ('bc4cba6d9fe8', 3,  8,  3, True),
    ('6c4cba6d9fe8', 3,  8, 18, True),
    ('0c4dba6d9fe8', 3, 13,  3, True),
    ('f04cba6d9fe8', 3, 13, 18, True),
]

sensor_df = pd.DataFrame(SENSORS, columns=['deviceId','col','shelf','row','has_co2'])
sensor_df['x_mm'] = sensor_df['col'].map(COL_X)
sensor_df['y_mm'] = sensor_df['row'].apply(ROW_Y)
sensor_df['z_mm'] = sensor_df['shelf'].apply(SHELF_Z)
sensor_df


## 3. Microclimate Assignment — Anisotropic IDW

### Rationale
Sensors are only in columns 1 and 3. Trays in columns 2 and 4 have no direct measurement.  
We interpolate hourly T, RH, CO₂, and VPD to each tray position using **Inverse Distance Weighting (IDW)**  
with **anisotropic scaling**: vertical (shelf) distance is weighted α = 3× more than horizontal,  
reflecting stronger thermal stratification in the vertical direction.

$$d_{eff} = \sqrt{\Delta x^2 + \Delta y^2 + (\alpha \cdot \Delta z)^2}$$

$$w_i = \frac{1/d_{eff,i}^{\,2}}{\sum_j 1/d_{eff,j}^{\,2}}$$

A humidity bias of **−7 pp** is applied to all column-4 trays post-interpolation.


In [ ]:
ALPHA_Z       = 3.0
IDW_POWER     = 2
COL4_H_OFFSET = -7.0

def compute_idw_weights(tray_col, tray_shelf, tray_row, variable='thv'):
    tx, ty, tz = COL_X[tray_col], ROW_Y(tray_row), SHELF_Z(tray_shelf)
    weights = {}
    for _, s in sensor_df.iterrows():
        if variable == 'co2' and not s['has_co2']:
            continue
        dx, dy = tx - s['x_mm'], ty - s['y_mm']
        dz = ALPHA_Z * (tz - s['z_mm'])
        dist = np.sqrt(dx**2 + dy**2 + dz**2)
        if dist == 0:
            return {s['deviceId']: 1.0}
        weights[s['deviceId']] = 1.0 / dist**IDW_POWER
    total = sum(weights.values())
    return {k: v/total for k, v in weights.items()}

# Coverage check
valid = mc.copy()
print("Microclimate coverage:")
for col in ['air_temp_c','air_humidity_pct','co2_ppm','vpd_kPa','ec_mS','ph','water_temp_c']:
    pct = valid[col].notna().mean() * 100
    print(f"  {col:<22}: {pct:.1f}%")
print()
valid[['air_temp_c','vpd_kPa','co2_ppm','ec_mS','ph','water_temp_c']].describe().round(2)


## 4. Empirical Yield Models

Two complementary models are fitted to `tray_microclimate.csv`:

| Model | Strengths | Validation |
|---|---|---|
| **OLS regression** | Interpretable coefficients, VIF diagnostics | R² on full dataset |
| **Random Forest** | Non-linear interactions, handles collinearity | Leave-One-Season-Out CV |

### Key finding
DLI (Daily Light Integral) is the dominant predictor (RF MDI = 0.37), consistent with the  
PPFD 180→310 µmol/m²/s transition causing a ~2.5× yield increase (705 g → 1 753 g mean).  
Within each PPFD regime, EC, water temperature, and VPD drive the remaining variance.


In [ ]:
# ── OLS: standardised coefficients + VIF diagnostics
features_ols = ['dli','growth_days','vpd_kPa','air_temp_c','co2_ppm',
                'air_humidity_pct','ec_mS','ph','water_temp_c','Shelf','Column']

sub    = valid[['yield_g'] + features_ols].dropna()
X_raw  = sub[features_ols]
X_std  = sm.add_constant((X_raw - X_raw.mean()) / X_raw.std())
ols    = sm.OLS(sub['yield_g'], X_std).fit()

print(f"OLS  R² = {ols.rsquared:.3f}  adj-R² = {ols.rsquared_adj:.3f}"
      f"  RMSE = {np.sqrt(ols.mse_resid):.1f} g  n = {len(sub)}")
print()

# Coefficient table
res_df = pd.DataFrame({'coef': ols.params, 'p': ols.pvalues}).drop('const')
res_df['sig'] = res_df['p'].apply(lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')))
res_df = res_df.sort_values('coef', key=abs, ascending=False)
print("Standardised OLS coefficients:")
print(res_df.round(4).to_string())
print()

# VIF check
vif_df = pd.DataFrame({
    'feature': features_ols,
    'VIF': [variance_inflation_factor(X_raw.values, i) for i in range(X_raw.shape[1])]
}).sort_values('VIF', ascending=False)
print("Variance Inflation Factors (VIF):")
print(vif_df.round(2).to_string(index=False))


In [ ]:
# ── OLS residual diagnostics (4-panel)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
resid = ols.resid
fitted = ols.fittedvalues

# 1. Residuals vs Fitted
axes[0,0].scatter(fitted, resid, alpha=0.15, s=4, color='#374151')
axes[0,0].axhline(0, color='k', lw=1)
axes[0,0].set_xlabel('Fitted values (g)'); axes[0,0].set_ylabel('Residuals (g)')
axes[0,0].set_title('Residuals vs Fitted')

# 2. Q-Q plot
(osm, osr), (slope, intercept, r) = stats.probplot(resid, dist='norm')
axes[0,1].scatter(osm, osr, alpha=0.3, s=6, color='#374151')
axes[0,1].plot([osm.min(), osm.max()],
               [slope*osm.min()+intercept, slope*osm.max()+intercept],
               'r-', lw=1.5)
axes[0,1].set_xlabel('Theoretical quantiles'); axes[0,1].set_ylabel('Sample quantiles')
axes[0,1].set_title(f'Normal Q-Q  (Shapiro-Wilk p={stats.shapiro(resid[:5000])[1]:.4f})')

# 3. Scale-Location
sqrt_abs_resid = np.sqrt(np.abs(resid))
axes[1,0].scatter(fitted, sqrt_abs_resid, alpha=0.15, s=4, color='#374151')
axes[1,0].set_xlabel('Fitted values (g)'); axes[1,0].set_ylabel('√|Residuals|')
axes[1,0].set_title('Scale-Location (Homoscedasticity check)')

# 4. Residual distribution
axes[1,1].hist(resid, bins=60, color='#4C9BE8', edgecolor='white', linewidth=0.3)
axes[1,1].set_xlabel('Residual (g)'); axes[1,1].set_ylabel('Count')
axes[1,1].set_title(f'Residual distribution  (skew={stats.skew(resid):.2f})')
xr = np.linspace(resid.min(), resid.max(), 200)
ax2t = axes[1,1].twinx()
ax2t.plot(xr, stats.norm.pdf(xr, resid.mean(), resid.std()),
          'r-', lw=1.5, label='Normal fit')
ax2t.set_ylabel('Density'); ax2t.legend(loc='upper right')

plt.suptitle('OLS Yield Model — Residual Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Random Forest: Leave-One-Season-Out Cross-Validation (LOSO-CV)
features_rf = ['dli','growth_days','vpd_kPa','air_temp_c','co2_ppm',
               'air_humidity_pct','ec_mS','ph','water_temp_c','Shelf','Column','Row']

df_rf      = valid[['Season','yield_g','ppfd'] + features_rf].dropna()
seasons    = sorted(df_rf['Season'].unique())
n_seasons  = len(seasons)

oof_pred = np.full(len(df_rf), np.nan)
oof_idx  = {}

print(f"LOSO-CV over {n_seasons} seasons …")
for i, s_out in enumerate(seasons):
    tr_idx = df_rf.index[df_rf['Season'] != s_out]
    te_idx = df_rf.index[df_rf['Season'] == s_out]
    rf_cv  = RandomForestRegressor(n_estimators=200, max_features='sqrt',
                                    min_samples_leaf=15, random_state=42, n_jobs=-1)
    rf_cv.fit(df_rf.loc[tr_idx, features_rf], df_rf.loc[tr_idx, 'yield_g'])
    oof_pred[df_rf.index.get_indexer(te_idx)] = rf_cv.predict(df_rf.loc[te_idx, features_rf])
    oof_idx[s_out] = {'r2': r2_score(df_rf.loc[te_idx,'yield_g'],
                                      rf_cv.predict(df_rf.loc[te_idx, features_rf]))}
    print(f"  [{i+1:2d}/{n_seasons}] Season {s_out}  "
          f"n={len(te_idx):4d}  R²={oof_idx[s_out]['r2']:.3f}")

# Final model on all data for feature importance
rf_full = RandomForestRegressor(n_estimators=300, max_features='sqrt',
                                  min_samples_leaf=10, random_state=42, n_jobs=-1)
rf_full.fit(df_rf[features_rf], df_rf['yield_g'])

r2_loso  = r2_score(df_rf['yield_g'], oof_pred)
rmse_loso = np.sqrt(mean_squared_error(df_rf['yield_g'], oof_pred))
print()
print(f"LOSO-CV  R² = {r2_loso:.3f}  |  RMSE = {rmse_loso:.1f} g  |  n = {len(df_rf)}")


In [ ]:
# ── RF: Feature importance (MDI + Permutation) + OOF predicted vs actual
imp_mdi = pd.Series(rf_full.feature_importances_, index=features_rf)

perm    = permutation_importance(rf_full, df_rf[features_rf], df_rf['yield_g'],
                                  n_repeats=10, random_state=42, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=features_rf)
imp_perm_std = pd.Series(perm.importances_std, index=features_rf)

# Sort by MDI
order   = imp_mdi.sort_values(ascending=True).index

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# MDI importance
axes[0].barh(range(len(order)), imp_mdi[order], height=0.6, color='#4C9BE8', alpha=0.85)
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(order, fontsize=8)
axes[0].set_xlabel('MDI Importance'); axes[0].set_title('RF — MDI Feature Importance')

# Permutation importance
axes[1].barh(range(len(order)), imp_perm[order], height=0.6,
             xerr=imp_perm_std[order], color='#F4A020', alpha=0.85, ecolor='#374151', capsize=2)
axes[1].set_yticks(range(len(order))); axes[1].set_yticklabels(order, fontsize=8)
axes[1].set_xlabel('Permutation Importance (±1 SD)'); axes[1].set_title('RF — Permutation Importance')

# OOF predicted vs actual
for ppfd, grp in df_rf.groupby('ppfd'):
    idx = grp.index
    axes[2].scatter(grp['yield_g'], oof_pred[df_rf.index.get_indexer(idx)],
                    color=PPFD_COLORS[ppfd], alpha=0.15, s=5, label=f'PPFD {int(ppfd)}')
axes[2].plot([0,3400],[0,3400],'k--',lw=1)
axes[2].set_xlabel('Actual yield (g/tray)'); axes[2].set_ylabel('OOF predicted (g/tray)')
axes[2].set_title(f'LOSO-CV  OOF R² = {r2_loso:.3f} | RMSE = {rmse_loso:.0f} g')
axes[2].legend(markerscale=3)

plt.suptitle('Random Forest — Full Model Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Per-season LOSO-CV R² — identify weak/strong seasons
season_r2 = pd.DataFrame(oof_idx).T.reset_index()
season_r2.columns = ['Season','R2_LOSO']
season_r2 = season_r2.sort_values('R2_LOSO')

fig, ax = plt.subplots(figsize=(12, 4))
colors = [COL_VIABLE if r >= 0.4 else (COL_MARGINAL if r >= 0.2 else COL_LOSS)
          for r in season_r2['R2_LOSO']]
ax.bar(range(len(season_r2)), season_r2['R2_LOSO'], color=colors, width=0.8)
ax.axhline(r2_loso, color='#374151', lw=1.5, ls='--', label=f'Overall LOSO R²={r2_loso:.3f}')
ax.set_xticks(range(len(season_r2)))
ax.set_xticklabels(season_r2['Season'], rotation=90, fontsize=7)
ax.set_ylabel('Hold-out R²')
ax.set_title('Per-Season Leave-One-Out R² — Random Forest')
ax.legend()
plt.tight_layout(); plt.show()

print(f"Seasons with R² ≥ 0.4 : {(season_r2['R2_LOSO'] >= 0.4).sum()}")
print(f"Seasons with R² < 0.2 : {(season_r2['R2_LOSO'] < 0.2).sum()}")


## 4.3 TabPFN — Tabular Foundation Model (LOSO-CV)

**TabPFN** is a pre-trained transformer for tabular regression. Unlike OLS and Random Forest,
it uses *in-context learning*: no training loop, no hyperparameter tuning.
It is pre-trained on millions of synthetic datasets and typically outperforms classical
ML on small-to-medium tabular problems.

We evaluate it under the same **Leave-One-Season-Out (LOSO-CV)** protocol as the Random Forest
so results are directly comparable.

In [ ]:
# ── TabPFN: install if needed + LOSO-CV
try:
    from tabpfn import TabPFNRegressor
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tabpfn', '-q'])
    from tabpfn import TabPFNRegressor

# Same features and dataset as Random Forest section
features_rf = ['dli','growth_days','vpd_kPa','air_temp_c','co2_ppm',
               'air_humidity_pct','ec_mS','ph','water_temp_c','Shelf','Column','Row']

df_rf = valid[['Season','yield_g','ppfd'] + features_rf].dropna()
seasons = sorted(df_rf['Season'].unique())

oof_pred_tpfn = np.full(len(df_rf), np.nan)
oof_idx_tpfn  = {}

print(f'TabPFN LOSO-CV over {len(seasons)} seasons ...')
for i, s_out in enumerate(seasons):
    tr_idx = df_rf.index[df_rf['Season'] != s_out]
    te_idx = df_rf.index[df_rf['Season'] == s_out]

    X_tr = df_rf.loc[tr_idx, features_rf].values
    y_tr = df_rf.loc[tr_idx, 'yield_g'].values
    X_te = df_rf.loc[te_idx, features_rf].values
    y_te = df_rf.loc[te_idx, 'yield_g'].values

    tpfn = TabPFNRegressor(device='cpu', ignore_pretraining_limits=True)
    tpfn.fit(X_tr, y_tr)
    preds = tpfn.predict(X_te)

    oof_pred_tpfn[df_rf.index.get_indexer(te_idx)] = preds
    r2_s = r2_score(y_te, preds)
    oof_idx_tpfn[s_out] = {'r2': r2_s}
    print(f'  [{i+1:2d}/{len(seasons)}] Season {s_out}  n={len(te_idx):4d}  R²={r2_s:.3f}')

r2_tpfn   = r2_score(df_rf['yield_g'], oof_pred_tpfn)
rmse_tpfn = np.sqrt(mean_squared_error(df_rf['yield_g'], oof_pred_tpfn))
print()
print(f'TabPFN  LOSO-CV  R² = {r2_tpfn:.3f}  |  RMSE = {rmse_tpfn:.1f} g  |  n = {len(df_rf)}')

In [ ]:
# ── Model comparison: OLS vs Random Forest vs TabPFN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart: LOSO-CV R² comparison ---
models  = ['OLS
(full data)', 'Random Forest
(LOSO-CV)', 'TabPFN
(LOSO-CV)']
r2_vals = [ols.rsquared, r2_loso, r2_tpfn]
colors  = ['#374151', '#F4A020', '#4C9BE8']

bars = axes[0].bar(models, r2_vals, color=colors, width=0.5, alpha=0.85)
for bar, val in zip(bars, r2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('R²')
axes[0].set_title('Model Comparison — R²
(LOSO-CV = held-out seasons)')
axes[0].axhline(0.5, color='k', lw=0.8, ls='--', alpha=0.4, label='R²=0.5 reference')
axes[0].legend(fontsize=8)

# --- OOF predicted vs actual: RF vs TabPFN ---
for ppfd, grp in df_rf.groupby('ppfd'):
    idx = df_rf.index.get_indexer(grp.index)
    axes[1].scatter(grp['yield_g'], oof_pred_tpfn[idx],
                    color=PPFD_COLORS[ppfd], alpha=0.2, s=5, label=f'PPFD {int(ppfd)}')
axes[1].plot([0, 3400], [0, 3400], 'k--', lw=1)
axes[1].set_xlabel('Actual yield (g/tray)')
axes[1].set_ylabel('TabPFN predicted (g/tray)')
axes[1].set_title(f'TabPFN — OOF Predicted vs Actual
LOSO-CV R² = {r2_tpfn:.3f}  |  RMSE = {rmse_tpfn:.0f} g')
axes[1].legend(markerscale=3, fontsize=8)

plt.suptitle('TabPFN vs Classical Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
print('
=== MODEL COMPARISON SUMMARY ===')
print(f'{"Model":<30} {"R²":>8} {"RMSE (g)":>12} {"Validation"}')
print('-'*65)
print(f'{"OLS Regression":<30} {ols.rsquared:>8.3f} {np.sqrt(ols.mse_resid):>12.1f} Full dataset')
print(f'{"Random Forest (LOSO-CV)":<30} {r2_loso:>8.3f} {rmse_loso:>12.1f} Leave-One-Season-Out')
print(f'{"TabPFN (LOSO-CV)":<30} {r2_tpfn:>8.3f} {rmse_tpfn:>12.1f} Leave-One-Season-Out')

## 5. Biomass Growth Curves (Gompertz)

### Model
$$W(t) = A \cdot \exp\!\left(-\exp(-k\,(t - t_i))\right)$$

| Parameter | Meaning |
|---|---|
| A | Asymptote — maximum biomass (g/sample) |
| k | Growth rate constant (day⁻¹) |
| t_i | Inflection day (maximum growth rate) |

### Key results
- Mean fit R² = **0.998** across 58 seasons  
- PPFD 180 → k̄ = 0.128/day; PPFD 310 → k̄ = 0.166/day  (+30%)  
- Only DLI significantly predicts k (r = +0.38, p = 0.003)


In [ ]:
avg_raw = pd.read_csv(f"{UPLOADS}Average data per season - Average data per season.csv")
bm_cols = ['Weight D5 (g)','Weight D10 (g)','Weight D15 (g)','Weight D20 (g)']
days_m  = [5, 10, 15, 20]

def gompertz(t, A, k, ti):
    return A * np.exp(-np.exp(-k * (t - ti)))

gompertz_results = []
for _, row in avg_raw.iterrows():
    pts = [(d, row[f'Weight D{d} (g)']) for d in days_m
           if pd.notna(row[f'Weight D{d} (g)']) and row[f'Weight D{d} (g)'] > 0]
    if len(pts) < 3:
        continue
    t_arr = np.array([p[0] for p in pts], dtype=float)
    w_arr = np.array([p[1] for p in pts], dtype=float)
    try:
        popt, pcov = curve_fit(gompertz, t_arr, w_arr,
                               p0=[w_arr.max()*1.5, 0.3, 12.0],
                               bounds=([0,0.01,0],[100,5,40]), maxfev=5000)
        A_fit, k_fit, ti_fit = popt
        perr = np.sqrt(np.diag(pcov))
        r2 = 1 - (np.sum((w_arr - gompertz(t_arr,*popt))**2) /
                  np.sum((w_arr-w_arr.mean())**2))
        gompertz_results.append({
            'Season': row['Harvest'], 'PPFD': row['PPFD (µmol/m²/s)'],
            'DLI': row['DLI (mol/m²/day)'], 'A': A_fit, 'k': k_fit, 'ti': ti_fit,
            'k_se': perr[1], 'r2': r2, 'Yield_g': row['Avg Yield/Tray (g)']
        })
    except Exception:
        pass

gom_df = pd.DataFrame(gompertz_results)
print(f"Seasons fitted: {len(gom_df)}  |  Mean R² = {gom_df['r2'].mean():.4f}")
print()

# Per-PPFD summary with bootstrap 95% CI on k
for ppfd, grp in gom_df.groupby('PPFD'):
    k_vals = grp['k'].values
    ci = stats.t.interval(0.95, len(k_vals)-1, loc=k_vals.mean(), scale=stats.sem(k_vals))
    print(f"PPFD {ppfd:>4.0f}  n={len(grp):2d}  k̄={k_vals.mean():.4f} "
          f"[95%CI {ci[0]:.4f}–{ci[1]:.4f}]  Ā={grp['A'].mean():.2f}  "
          f"t̄ᵢ={grp['ti'].mean():.2f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# k vs DLI with regression CI
ax = axes[0]
for ppfd, grp in gom_df.groupby('PPFD'):
    ax.scatter(grp['DLI'], grp['k'], color=PPFD_COLORS[ppfd],
               alpha=0.8, s=60, label=f'PPFD {int(ppfd)}')
p_lin = np.polyfit(gom_df['DLI'], gom_df['k'], 1)
xr    = np.linspace(gom_df['DLI'].min(), gom_df['DLI'].max(), 100)
r_val, p_val = stats.pearsonr(gom_df['DLI'], gom_df['k'])
ax.plot(xr, np.polyval(p_lin, xr), 'k--', lw=1.5, label=f'r={r_val:.2f}, p={p_val:.3f}')
ax.set_xlabel('DLI (mol/m²/day)'); ax.set_ylabel('Gompertz k (day⁻¹)')
ax.set_title('Growth rate k vs DLI'); ax.legend(fontsize=8)

# k distribution by PPFD
ax = axes[1]
ax.hist(gom_df[gom_df['PPFD']==180]['k'], bins=15, color='#4C9BE8', alpha=0.65, label='PPFD 180')
ax.hist(gom_df[gom_df['PPFD']==310]['k'], bins=10, color='#F4A020', alpha=0.65, label='PPFD 310')
ax.set_xlabel('k (day⁻¹)'); ax.set_ylabel('Count')
ax.set_title('Distribution of growth rate k by PPFD'); ax.legend()

# A (asymptote) vs Yield at harvest
ax = axes[2]
for ppfd, grp in gom_df.groupby('PPFD'):
    ax.scatter(grp['A'], grp['Yield_g'], color=PPFD_COLORS[ppfd],
               alpha=0.8, s=50, label=f'PPFD {int(ppfd)}')
ax.set_xlabel('Gompertz A (asymptote, g)'); ax.set_ylabel('Observed yield (g/tray)')
ax.set_title('Asymptote vs Observed Harvest Yield')
ax.legend(); ax.plot([0,50],[0,50],'k--',lw=0.8)

plt.suptitle('Gompertz Growth Model', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 6. Mechanistic Growth Model (Vanthoor, simplified)

### Framework
Adapted from Vanthoor *et al.* (2011). Simplified single-state equation for microgreens:

$$\frac{dW}{dt} = \varepsilon_{LUE} \cdot I_{PAR} \cdot f_{int}(LAI) \cdot L_h - R_m \cdot W \cdot Q_{10}^{(T-20)/10}$$

| Symbol | Meaning | Value |
|---|---|---|
| ε_LUE | Light use efficiency (g DM mol⁻¹ PAR) | **1.673** (calibrated) |
| K | Extinction coefficient | 0.80 (literature) |
| SLA | Specific leaf area (m² g⁻¹) | 0.028 (literature) |
| Rm | Maintenance respiration (g DW g⁻¹ day⁻¹) | 0.012 (literature) |
| DM% | Dry matter fraction | **6%** (assumed) |

### Calibration notes
- Initialised at **D5** (cotyledon phase D0–D5 not modelled — driven by seed reserves)  
- ε_LUE found by **grid search** minimising log-MSE over calibration dataset  
- To refine: oven-dry ~10 plants (65 °C, 48 h) → better DM% → re-calibrate ε_LUE  
- MAPE 31.3% is consistent with missing DM% measurement and nutrient-solution variability


In [ ]:
TRAY_AREA   = 1.155**2   # m²
DM_FRACTION = 0.06
EPS_LUE     = 1.673
K_EXT       = 0.80
SLA         = 0.028
RM          = 0.012

def vanthoor_simulate(row, eps=EPS_LUE, dm=DM_FRACTION):
    if pd.isna(row.get('Weight D5 (g)')) or row['Weight D5 (g)'] <= 0:
        return {}
    plants = row.get('plants_per_tray', 900)
    DW0    = row['Weight D5 (g)'] * dm * plants / TRAY_AREA
    T      = row['Avg Temp (°C)']
    PPFD   = row['PPFD (µmol/m²/s)']
    lh     = row['Light h/day']
    Q10f   = 2.0**((T - 20) / 10)
    PAR_h  = PPFD * 1e-6 * 3600
    DW, out = DW0, {5: DW0}
    for d in range(6, int(row['Growing Days']) + 1):
        f_int = 1 - np.exp(-K_EXT * SLA * DW)
        dDW   = eps * PAR_h * f_int * lh - RM * DW * Q10f
        DW    = max(DW + dDW, 0.0)
        out[d] = DW
    return out

# Merge plants_per_tray and validate
ppts = prod.groupby('Season')['plants_per_tray'].first().reset_index()
ppts.columns = ['Harvest','plants_per_tray']
avg2 = avg_raw.merge(ppts, on='Harvest', how='left')
avg2['plants_per_tray'] = avg2['plants_per_tray'].fillna(900)

y_obs, y_pred_v = [], []
for _, row in avg2.iterrows():
    for d in [10, 15, 20]:
        fw = row.get(f'Weight D{d} (g)')
        if (pd.notna(fw) and fw > 0 and
            pd.notna(row.get('Weight D5 (g)')) and row['Weight D5 (g)'] > 0):
            dw_obs = fw * DM_FRACTION * row['plants_per_tray'] / TRAY_AREA
            sim    = vanthoor_simulate(row)
            if d in sim:
                y_obs.append(dw_obs); y_pred_v.append(sim[d])

y_obs   = np.array(y_obs); y_pred_v = np.array(y_pred_v)
r2_v    = r2_score(y_obs, y_pred_v)
mape_v  = np.mean(np.abs(y_obs - y_pred_v) / y_obs) * 100
rmse_v  = np.sqrt(mean_squared_error(y_obs, y_pred_v))

print(f"Vanthoor (ε_LUE={EPS_LUE}, DM={DM_FRACTION*100:.0f}%)")
print(f"  R²   = {r2_v:.3f}")
print(f"  RMSE = {rmse_v:.2f} g DM/m²")
print(f"  MAPE = {mape_v:.1f}%")
print(f"  n    = {len(y_obs)} points (D10–D20)")


In [ ]:
# ── ε_LUE grid search curve
eps_grid   = np.linspace(0.5, 3.5, 120)
mape_grid  = []

for eps in eps_grid:
    y_p = []
    y_o = []
    for _, row in avg2.iterrows():
        for d in [10, 15, 20]:
            fw = row.get(f'Weight D{d} (g)')
            if pd.notna(fw) and fw > 0 and pd.notna(row.get('Weight D5 (g)')) and row['Weight D5 (g)'] > 0:
                dw_o = fw * DM_FRACTION * row['plants_per_tray'] / TRAY_AREA
                sim  = vanthoor_simulate(row, eps=eps)
                if d in sim:
                    y_o.append(dw_o); y_p.append(sim[d])
    y_o = np.array(y_o); y_p = np.array(y_p)
    mape_grid.append(np.mean(np.abs(y_o - y_p) / y_o) * 100)

best_eps = eps_grid[np.argmin(mape_grid)]
print(f"Best ε_LUE = {best_eps:.3f} g DM/mol PAR  (MAPE={min(mape_grid):.1f}%)")

# ── Plot: calibration curve + predicted vs actual + trajectories
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# eps sweep
axes[0].plot(eps_grid, mape_grid, color='#4C9BE8', lw=2)
axes[0].axvline(best_eps, color=COL_LOSS, lw=1.5, ls='--',
                label=f'ε_LUE={best_eps:.3f}')
axes[0].set_xlabel('ε_LUE (g DM / mol PAR)')
axes[0].set_ylabel('MAPE (%)')
axes[0].set_title('Vanthoor calibration — ε_LUE grid search')
axes[0].legend()

# Predicted vs actual
for _, row in avg2.iterrows():
    ppfd = row['PPFD (µmol/m²/s)']
    sim  = vanthoor_simulate(row)
    for d in [10, 15, 20]:
        fw = row.get(f'Weight D{d} (g)')
        if pd.notna(fw) and fw > 0 and d in sim:
            dw_o = fw * DM_FRACTION * row['plants_per_tray'] / TRAY_AREA
            axes[1].scatter(dw_o, sim[d], color=PPFD_COLORS.get(ppfd,'gray'), alpha=0.55, s=20)

lim = [0, max(y_obs.max(), y_pred_v.max()) * 1.05]
axes[1].plot(lim, lim, 'k--', lw=1)
axes[1].set_xlabel('Observed DW/m² (g)'); axes[1].set_ylabel('Predicted DW/m² (g)')
axes[1].set_title(f'Predicted vs Observed (D10–D20)\nR²={r2_v:.3f}  MAPE={mape_v:.1f}%')
handles = [plt.scatter([],[],c=c,label=f'PPFD {p}') for p,c in PPFD_COLORS.items()]
axes[1].legend(handles=handles)

# Growth trajectories (8 example seasons)
sel = avg2[avg2[bm_cols].notna().sum(axis=1) >= 3].head(8)
for _, row in sel.iterrows():
    if pd.isna(row.get('Weight D5 (g)')) or row['Weight D5 (g)'] <= 0: continue
    sim = vanthoor_simulate(row)
    c   = PPFD_COLORS.get(row['PPFD (µmol/m²/s)'], 'gray')
    axes[2].plot(list(sim.keys()), list(sim.values()), color=c, alpha=0.6, lw=1.8)
    obs = {d: row[f'Weight D{d} (g)']*DM_FRACTION*row['plants_per_tray']/TRAY_AREA
           for d in days_m if pd.notna(row.get(f'Weight D{d} (g)')) and row[f'Weight D{d} (g)'] > 0}
    axes[2].scatter(list(obs.keys()), list(obs.values()), color=c, s=45, zorder=5)

axes[2].set_xlabel('Days after growth start'); axes[2].set_ylabel('DW/m² (g)')
axes[2].set_title('Growth trajectories: model (lines) vs observations (dots)')
axes[2].legend(handles=[plt.Line2D([0],[0],color=c,lw=2,label=f'PPFD {p}')
                         for p,c in PPFD_COLORS.items()])

plt.suptitle('Vanthoor Mechanistic Model — Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 7. Summary of Calibrated Parameters & Model Performance

| Model | Purpose | Metric | Value |
|---|---|---|---|
| **OLS** | Interpretable yield regression | R² | 0.61 |
| **Random Forest (LOSO-CV)** | Non-linear yield prediction | CV R² | ~0.50 |
| **Gompertz** | Growth dynamics per season | Mean R² | 0.998 |
| **Vanthoor** | Mechanistic DW accumulation (D5→harvest) | R² / MAPE | 0.50 / 31.3% |

### Vanthoor calibrated constants

| Parameter | Symbol | Value | Source |
|---|---|---|---|
| Light use efficiency | ε_LUE | 1.673 g DM mol⁻¹ PAR | grid-search calibration |
| Extinction coefficient | K | 0.80 | literature (leafy greens) |
| Specific leaf area | SLA | 0.028 m² g⁻¹ | literature |
| Maintenance respiration | Rm | 0.012 day⁻¹ | literature |
| DM fraction | — | 6% | **assumed** — refine with oven-dry measurement |

---


## 8. Financial Bridge Module — R/m²/year Viability Analysis

### The bridge metric
This section connects the crop model output to financial viability using a single unit-economics equation:

$$R/\text{m}^2/\text{year} = Y \cdot P - \text{OPEX} - \text{CAPEX}_{amort}$$

where:
- **Y** [kg/m²/year] — annualised fresh-weight yield (from crop model)  
- **P** [USD/kg] — farm-gate price  
- **OPEX** [USD/m²/year] — operating costs (energy, labour, nutrients, water)  
- **CAPEX_amort** [USD/m²/year] — annualised capital expenditure  

The **break-even yield** required for viability:

$$Y^* = \frac{\text{OPEX} + \text{CAPEX}_{amort}}{P}$$

### Default parameters (literature benchmarks — CEA spinach / microgreens)
These are reference values from published CEA techno-economic analyses.  
Adjust `OPEX_M2_YEAR`, `CAPEX_AMORT`, and `P_SCENARIOS` to match Phyllome's actual cost structure.

| Parameter | Value | Source |
|---|---|---|
| Tray area | 1.334 m² (1.155 × 1.155) | Phyllome measured |
| Mean growth days | ~18 days | Phyllome data |
| OPEX | 120 USD/m²/year | Al-Chalabi (2015); Benke & Tomkins (2017) |
| CAPEX amort | 45 USD/m²/year | Graamans et al. (2018); assumes 10-yr life |
| Baseline price P | 8 USD/kg | Premium spinach wholesale (literature range 4–20) |


In [ ]:
# ── Financial bridge parameters
TRAY_AREA_M2   = 1.155**2        # m²
DM_FRAC        = 0.06
OPEX_M2_YEAR   = 120.0           # USD / m² / year  (literature baseline)
CAPEX_AMORT    = 45.0            # USD / m² / year  (10-yr amortised CAPEX)
TOTAL_COST     = OPEX_M2_YEAR + CAPEX_AMORT

P_BASE         = 8.0             # USD / kg  (baseline farm-gate price)

# ── Annualised yield from tray_microclimate.csv
# cycles/year estimated from observed growth_days
mean_gd        = mc['growth_days'].mean()
cycles_year    = 365.0 / mean_gd

mc['yield_kg_cycle_m2'] = mc['yield_g'] / 1000 / TRAY_AREA_M2
mc['Y_kg_m2_year']      = mc['yield_kg_cycle_m2'] * cycles_year

# ── Bridge metric at baseline price
mc['R_m2_year']  = mc['Y_kg_m2_year'] * P_BASE - TOTAL_COST

# ── Break-even yield
Y_star = TOTAL_COST / P_BASE

print("=== FINANCIAL BRIDGE — SUMMARY ===")
print(f"  Tray area         : {TRAY_AREA_M2:.3f} m²")
print(f"  Mean growth days  : {mean_gd:.1f} d  →  {cycles_year:.1f} cycles/year")
print()
print(f"  OPEX              : {OPEX_M2_YEAR:.0f} USD/m²/year")
print(f"  CAPEX_amort       : {CAPEX_AMORT:.0f} USD/m²/year")
print(f"  Total cost        : {TOTAL_COST:.0f} USD/m²/year")
print()
print(f"  Baseline price P  : {P_BASE:.0f} USD/kg")
print(f"  Break-even Y*     : {Y_star:.2f} kg/m²/year")
print()
print("Observed yield distribution:")
print(mc['Y_kg_m2_year'].describe().round(2).rename({
    'mean':'mean Y (kg/m²/yr)', 'std':'std', '50%':'median',
    'min':'min', 'max':'max'}).to_string())
print()
print(f"  Mean R/m²/year    : {mc['R_m2_year'].mean():.2f} USD")
print(f"  Median R/m²/year  : {mc['R_m2_year'].median():.2f} USD")
print(f"  Fraction viable   : {(mc['R_m2_year']>0).mean()*100:.1f}%")


In [ ]:
# ── Financial bridge visualisations
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Y distribution by PPFD vs Y*
ax = axes[0, 0]
for ppfd, grp in mc.groupby('ppfd'):
    ax.hist(grp['Y_kg_m2_year'], bins=40, alpha=0.6,
            color=PPFD_COLORS[ppfd], label=f'PPFD {int(ppfd)}', density=True)
ax.axvline(Y_star, color=COL_LOSS, lw=2, ls='--',
           label=f'Y* = {Y_star:.1f} kg/m²/yr  (P=${P_BASE}/kg)')
ax.set_xlabel('Annualised yield (kg/m²/year)'); ax.set_ylabel('Density')
ax.set_title('Yield distribution vs break-even Y*'); ax.legend(fontsize=8)

# 2. R/m²/year distribution
ax = axes[0, 1]
r_vals = mc['R_m2_year'].dropna()
ax.hist(r_vals[r_vals < 0], bins=40, color=COL_LOSS, alpha=0.7, label='Loss')
ax.hist(r_vals[r_vals >= 0], bins=40, color=COL_VIABLE, alpha=0.7, label='Viable')
ax.axvline(0, color='k', lw=1.5, ls='--')
ax.set_xlabel('R/m²/year (USD)'); ax.set_ylabel('Count')
ax.set_title(f'Revenue − Cost distribution  (P=${P_BASE}/kg)')
ax.legend()

# 3. Break-even chart: R/m²/year vs P for PPFD groups
ax = axes[1, 0]
P_range = np.linspace(2, 24, 200)
for ppfd, grp in mc.groupby('ppfd'):
    Y_med = grp['Y_kg_m2_year'].median()
    Y_q25 = grp['Y_kg_m2_year'].quantile(0.25)
    Y_q75 = grp['Y_kg_m2_year'].quantile(0.75)
    R_med = Y_med * P_range - TOTAL_COST
    R_q25 = Y_q25 * P_range - TOTAL_COST
    R_q75 = Y_q75 * P_range - TOTAL_COST
    c = PPFD_COLORS[ppfd]
    ax.plot(P_range, R_med, color=c, lw=2.5, label=f'PPFD {int(ppfd)} (median Y)')
    ax.fill_between(P_range, R_q25, R_q75, color=c, alpha=0.15, label='IQR')
ax.axhline(0, color='k', lw=1, ls='--')
ax.axvline(P_BASE, color='#374151', lw=1, ls=':', label=f'Baseline P=${P_BASE}')
ax.fill_between(P_range, 0, 1e6, where=(P_range >= 0),
                alpha=0.05, color=COL_VIABLE, transform=ax.get_xaxis_transform())
ax.set_xlim(P_range.min(), P_range.max())
ax.set_xlabel('Farm-gate price P (USD/kg)'); ax.set_ylabel('R/m²/year (USD)')
ax.set_title('Break-even chart: R/m²/year vs Price')
ax.legend(fontsize=8)

# 4. Viability heatmap: OPEX vs P (at observed median Y)
ax = axes[1, 1]
P_grid   = np.linspace(4, 20, 80)
O_grid   = np.linspace(60, 200, 80)
PP, OO   = np.meshgrid(P_grid, O_grid)
Y_global = mc['Y_kg_m2_year'].median()
RR       = Y_global * PP - OO - CAPEX_AMORT

contf = ax.contourf(PP, OO, RR, levels=20, cmap='RdYlGn')
ax.contour(PP, OO, RR, levels=[0], colors='k', linewidths=2)
plt.colorbar(contf, ax=ax, label='R/m²/year (USD)')
ax.scatter([P_BASE], [OPEX_M2_YEAR], color='white', s=120, zorder=5,
           edgecolors='k', label=f'Baseline (P={P_BASE}, OPEX={OPEX_M2_YEAR})')
ax.set_xlabel('Price P (USD/kg)'); ax.set_ylabel('OPEX (USD/m²/year)')
ax.set_title(f'Viability heatmap (median Y={Y_global:.1f} kg/m²/yr,\nblack line = break-even)')
ax.legend(fontsize=8)

plt.suptitle('Financial Bridge — R/m²/year Viability Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Monte Carlo: propagate uncertainty in Y and P → distribution of R/m²/year
np.random.seed(42)
N_MC = 50_000

# Y distribution (log-normal fit to observed data)
Y_obs    = mc['Y_kg_m2_year'].dropna()
Y_mu     = np.log(Y_obs).mean()
Y_sigma  = np.log(Y_obs).std()
Y_mc     = np.random.lognormal(Y_mu, Y_sigma, N_MC)

# P scenarios: baseline, optimistic, pessimistic
P_scenarios = {
    'Pessimistic (P=5)' : 5.0,
    'Baseline (P=8)'    : 8.0,
    'Optimistic (P=14)' : 14.0,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_mc = {}
for label, P_val in P_scenarios.items():
    # Add ±10% price uncertainty
    P_mc   = P_val * np.random.normal(1.0, 0.10, N_MC)
    R_mc   = Y_mc * P_mc - TOTAL_COST
    pv     = (R_mc > 0).mean() * 100
    results_mc[label] = {'P': P_val, 'R_mean': R_mc.mean(),
                          'R_p10': np.percentile(R_mc, 10),
                          'R_p50': np.percentile(R_mc, 50),
                          'R_p90': np.percentile(R_mc, 90),
                          'prob_viable': pv}
    c = COL_VIABLE if pv > 60 else (COL_MARGINAL if pv > 30 else COL_LOSS)
    axes[0].hist(R_mc, bins=80, density=True, alpha=0.55, color=c, label=f'{label}: P(R>0)={pv:.0f}%')

axes[0].axvline(0, color='k', lw=1.5, ls='--')
axes[0].set_xlabel('R/m²/year (USD)'); axes[0].set_ylabel('Density')
axes[0].set_title('Monte Carlo — R/m²/year under P and Y uncertainty\n(±10% price noise, log-normal Y)')
axes[0].legend(fontsize=8)

# Tornado chart: sensitivity of R to individual inputs
Y_base  = Y_obs.median()
R_base  = Y_base * P_BASE - TOTAL_COST
params  = {
    'Y (+10%)' : (Y_base * 1.1 * P_BASE - TOTAL_COST) - R_base,
    'Y (−10%)' : (Y_base * 0.9 * P_BASE - TOTAL_COST) - R_base,
    'P (+10%)' : (Y_base * P_BASE * 1.1 - TOTAL_COST) - R_base,
    'P (−10%)' : (Y_base * P_BASE * 0.9 - TOTAL_COST) - R_base,
    'OPEX (+10%)': (Y_base * P_BASE - TOTAL_COST * 1.1) - R_base,
    'OPEX (−10%)': (Y_base * P_BASE - TOTAL_COST * 0.9) - R_base,
    'Cycles (+1)' : ((Y_base / cycles_year) * (cycles_year + 1) * P_BASE - TOTAL_COST) - R_base,
    'Cycles (−1)' : ((Y_base / cycles_year) * (cycles_year - 1) * P_BASE - TOTAL_COST) - R_base,
}
labels_t = list(params.keys())
deltas   = list(params.values())
colors_t = [COL_VIABLE if d >= 0 else COL_LOSS for d in deltas]

axes[1].barh(range(len(labels_t)), deltas, color=colors_t, height=0.6)
axes[1].axvline(0, color='k', lw=1)
axes[1].set_yticks(range(len(labels_t))); axes[1].set_yticklabels(labels_t, fontsize=8)
axes[1].set_xlabel('ΔΔΔR/m²/year (USD)'); axes[1].set_title(f'Tornado chart (base R={R_base:.1f} USD/m²/yr)')

plt.suptitle('Financial Risk Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("Monte Carlo results:")
mc_res = pd.DataFrame(results_mc).T
print(mc_res[['P','R_p10','R_p50','R_p90','prob_viable']].round(2).to_string())


## 9. Operating Window Optimisation

### Goal
Identify the microclimate conditions that maximise yield per unit of resource consumed,  
particularly **yield per kWh** (energy efficiency) and **yield per litre of water**.  
This extends the financial bridge by revealing how to reach or exceed Y* at minimum cost.


In [ ]:
# ── Yield per kWh and optimal DLI window
# watts_per_tray is already in tray_microclimate.csv
mc_fin = mc.dropna(subset=['watts_per_tray','yield_g','dli','vpd_kPa','air_temp_c']).copy()

# Energy per cycle per tray  [kWh]
mc_fin['kwh_per_cycle'] = mc_fin['watts_per_tray'] * mc_fin['growth_days'] * 16 / 1000
# 16h assumed photoperiod average; adjust if actual data available

mc_fin['yield_per_kwh'] = mc_fin['yield_g'] / mc_fin['kwh_per_cycle']

print("Energy efficiency:")
for ppfd, grp in mc_fin.groupby('ppfd'):
    print(f"  PPFD {ppfd:.0f}: yield/kWh = {grp['yield_per_kwh'].median():.1f} g/kWh "
          f"(IQR {grp['yield_per_kwh'].quantile(0.25):.0f}–{grp['yield_per_kwh'].quantile(0.75):.0f})")
print()

# DLI bins → yield and yield/kWh
mc_fin['dli_bin'] = pd.cut(mc_fin['dli'], bins=np.arange(0, 25, 2.5),
                            labels=[f'{v:.1f}' for v in np.arange(1.25, 22.5, 2.5)])
dli_agg = mc_fin.groupby('dli_bin', observed=True)[['yield_g','yield_per_kwh']].agg(['mean','sem'])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# DLI vs yield
ax = axes[0]
ax.bar(range(len(dli_agg)), dli_agg['yield_g']['mean'],
       yerr=dli_agg['yield_g']['sem'], color='#4C9BE8', alpha=0.8,
       capsize=3, error_kw={'elinewidth':1})
ax.set_xticks(range(len(dli_agg)))
ax.set_xticklabels(dli_agg.index, rotation=45, fontsize=8)
ax.set_xlabel('DLI bin (mol/m²/day)'); ax.set_ylabel('Mean yield (g/tray)')
ax.set_title('Yield vs DLI bin (±SEM)')
ax.axhline(Y_star * TRAY_AREA_M2 * 1000 / cycles_year, color=COL_LOSS, ls='--',
           lw=1.5, label=f'Y* per cycle')
ax.legend(fontsize=8)

# DLI vs yield/kWh
ax = axes[1]
ax.bar(range(len(dli_agg)), dli_agg['yield_per_kwh']['mean'],
       yerr=dli_agg['yield_per_kwh']['sem'], color='#E9C46A', alpha=0.85,
       capsize=3, error_kw={'elinewidth':1})
ax.set_xticks(range(len(dli_agg)))
ax.set_xticklabels(dli_agg.index, rotation=45, fontsize=8)
ax.set_xlabel('DLI bin (mol/m²/day)'); ax.set_ylabel('Yield/kWh (g/kWh)')
ax.set_title('Energy efficiency vs DLI bin (±SEM)')

# Scatter: DLI vs R/m²/year coloured by temp
mc_fin2 = mc_fin.copy()
mc_fin2['R_m2_year'] = mc_fin2['Y_kg_m2_year'] * P_BASE - TOTAL_COST
sc = axes[2].scatter(mc_fin2['dli'], mc_fin2['R_m2_year'],
                      c=mc_fin2['air_temp_c'], cmap='RdYlGn_r',
                      alpha=0.2, s=5, vmin=18, vmax=26)
plt.colorbar(sc, ax=axes[2], label='Air temp (°C)')
axes[2].axhline(0, color='k', lw=1.5, ls='--', label='Break-even')
axes[2].set_xlabel('DLI (mol/m²/day)'); axes[2].set_ylabel('R/m²/year (USD)')
axes[2].set_title(f'DLI vs R/m²/year (P=${P_BASE}/kg)\ncolour = temperature')
axes[2].legend(fontsize=8)

plt.suptitle('Operating Window Optimisation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── Optimal operating conditions for Y ≥ Y* at P=P_BASE
viable = mc_fin[mc_fin['R_m2_year'] > 0] if 'R_m2_year' in mc_fin.columns else mc_fin[mc_fin['Y_kg_m2_year'] >= Y_star]

if len(viable) == 0:
    # Compute R if not present
    mc_fin['R_m2_year_base'] = mc_fin['Y_kg_m2_year'] * P_BASE - TOTAL_COST
    viable = mc_fin[mc_fin['R_m2_year_base'] > 0]

print(f"Viable trays at P=${P_BASE}/kg: {len(viable):,} / {len(mc_fin):,} ({len(viable)/len(mc_fin)*100:.1f}%)")
print()
print("Microclimate profile of VIABLE trays (mean ± std):")
for col, label in [('dli','DLI (mol/m²/day)'), ('air_temp_c','Temp (°C)'),
                    ('vpd_kPa','VPD (kPa)'), ('co2_ppm','CO₂ (ppm)'),
                    ('ec_mS','EC (mS/cm)'), ('ph','pH'),
                    ('water_temp_c','Water T (°C)'), ('growth_days','Growth days')]:
    if col not in viable.columns: continue
    mn, sd = viable[col].mean(), viable[col].std()
    mn_all, sd_all = mc_fin[col].mean(), mc_fin[col].std()
    print(f"  {label:<22}: viable {mn:.2f} ± {sd:.2f}  |  all {mn_all:.2f} ± {sd_all:.2f}")

print()
print("PPFD breakdown of viable trays:")
print(viable['ppfd'].value_counts().to_string())


## 10. Integrated Summary — Key Insights

### Crop model
| Model | CV R² | Notes |
|---|---|---|
| OLS regression | 0.61 | DLI, EC, water T dominant |
| Random Forest (LOSO-CV) | ~0.50 | Non-linear; DLI MDI importance = 0.37 |
| Gompertz | 0.998 (per-season) | k 30% higher at PPFD 310 |
| Vanthoor | R² = 0.50, MAPE = 31.3% | Refine with oven-dry DM measurement |

### Financial viability (literature benchmarks)
| Scenario | Y (kg/m²/yr) | R/m²/yr (USD) | P(viable) |
|---|---|---|---|
| All trays (mean) | ~14 | −52 | ~25% |
| PPFD 310 (mean) | ~26 | +43 | ~65% |
| Y* break-even (P=$8) | **20.6** | 0 | — |

### Key levers
1. **Increase PPFD to 310 µmol/m²/s** → most tractable path to cross Y*  
2. **Raise farm-gate price above $10.5/kg** → break-even at current mean yield  
3. **Reduce OPEX below $95/m²/yr** → break-even at current PPFD 310 yield  
4. **Optimise DLI** → target 15–22 mol/m²/day for best energy efficiency  

### Next steps
1. Measure DM fraction (oven-dry) → improve Vanthoor to MAPE < 15%  
2. Add actual Phyllome OPEX/CAPEX to replace literature benchmarks  
3. Run sensitivity on growth_days — faster cycles compound Y/year significantly  
4. Expand to additional crops (strawberry, saffron, chicory) using same framework  
